<a href="https://colab.research.google.com/github/simjonghyeon04/-/blob/main/%EC%A7%84%EC%A7%9C%20%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker # Changed import

SQLALCHEMY_DATABASE_URL = "sqlite:///./quotes.db"

engine = create_engine(
    SQLALCHEMY_DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

In [ ]:
from sqlalchemy import Column, Integer, String, Text, DateTime
from sqlalchemy.sql import func
from app.database import Base

class Quote(Base):
    __tablename__ = "quotes"
    __table_args__ = {'extend_existing': True}

    id = Column(Integer, primary_key=True, index=True)
    text = Column(Text, nullable=False)
    author = Column(String(100), nullable=False)
    tags = Column(String(500), nullable=True)  # 쉼표로 구분
    category = Column(String(100), nullable=True)
    created_at = Column(DateTime(timezone=True), server_default=func.now())
    updated_at = Column(DateTime(timezone=True), onupdate=func.now())

In [ ]:
get_ipython().system('mkdir -p app')
get_ipython().system('touch app/__init__.py')

In [ ]:
%%writefile app/database.py
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker # Changed import

SQLALCHEMY_DATABASE_URL = "sqlite:///./quotes.db"

engine = create_engine(
    SQLALCHEMY_DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

Overwriting app/database.py


In [ ]:
%%writefile app/models.py
from sqlalchemy import Column, Integer, String, Text, DateTime
from sqlalchemy.sql import func
from app.database import Base

class Quote(Base):
    __tablename__ = "quotes"
    __table_args__ = {'extend_existing': True} # Add this line

    id = Column(Integer, primary_key=True, index=True)
    text = Column(Text, nullable=False)
    author = Column(String(100), nullable=False)
    tags = Column(String(500), nullable=True)  # 쉼표로 구분
    category = Column(String(100), nullable=True)
    created_at = Column(DateTime(timezone=True), server_default=func.now())
    updated_at = Column(DateTime(timezone=True), onupdate=func.now())

Writing app/models.py


In [ ]:
%%writefile app/schemas.py
from pydantic import BaseModel
from datetime import datetime
from typing import Optional, List

class QuoteBase(BaseModel):
    text: str
    author: str
    tags: Optional[str] = None
    category: Optional[str] = None

class QuoteCreate(QuoteBase):
    pass

class QuoteUpdate(BaseModel):
    text: Optional[str] = None
    author: Optional[str] = None
    tags: Optional[str] = None
    category: Optional[str] = None

class QuoteResponse(QuoteBase):
    id: int
    created_at: datetime
    updated_at: Optional[datetime] = None

    class Config:
        from_attributes = True

class QuoteList(BaseModel):
    total: int
    quotes: List[QuoteResponse]

Writing app/schemas.py


In [ ]:
%%writefile app/crud.py
from sqlalchemy.orm import Session
from sqlalchemy import or_
from app.models import Quote
from app.schemas import QuoteCreate, QuoteUpdate
from typing import Optional, List

def create_quote(db: Session, quote: QuoteCreate) -> Quote:
    db_quote = Quote(**quote.model_dump())
    db.add(db_quote)
    db.commit()
    db.refresh(db_quote)
    return db_quote

def get_quote(db: Session, quote_id: int) -> Optional[Quote]:
    return db.query(Quote).filter(Quote.id == quote_id).first()

def get_quotes(
    db: Session,
    skip: int = 0,
    limit: int = 100,
    author: Optional[str] = None,
    category: Optional[str] = None,
    search: Optional[str] = None
) -> List[Quote]:
    query = db.query(Quote)

    if author:
        query = query.filter(Quote.author.ilike(f"%{author}%"))
    if category:
        query = query.filter(Quote.category == category)
    if search:
        query = query.filter(
            or_(
                Quote.text.ilike(f"%{search}%"),
                Quote.author.ilike(f"%{search}%"),
                Quote.tags.ilike(f"%{search}%")
            )
        )

    return query.offset(skip).limit(limit).all()

def get_quotes_count(
    db: Session,
    author: Optional[str] = None,
    category: Optional[str] = None,
    search: Optional[str] = None
) -> int:
    query = db.query(Quote)

    if author:
        query = query.filter(Quote.author.ilike(f"%{author}%"))
    if category:
        query = query.filter(Quote.category == category)
    if search:
        query = query.filter(
            or_(
                Quote.text.ilike(f"%{search}%"),
                Quote.author.ilike(f"%{search}%"),
                Quote.tags.ilike(f"%{search}%")
            )
        )

    return query.count()

def update_quote(db: Session, quote_id: int, quote: QuoteUpdate) -> Optional[Quote]:
    db_quote = db.query(Quote).filter(Quote.id == quote_id).first()
    if db_quote:
        update_data = quote.model_dump(exclude_unset=True)
        for key, value in update_data.items():
            setattr(db_quote, key, value)
        db.commit()
        db.refresh(db_quote)
    return db_quote

def delete_quote(db: Session, quote_id: int) -> bool:
    db_quote = db.query(Quote).filter(Quote.id == quote_id).first()
    if db_quote:
        db.delete(db_quote)
        db.commit()
        return True
    return False

def get_all_authors(db: Session) -> List[str]:
    results = db.query(Quote.author).distinct().all()
    return [r[0] for r in results]

def get_all_categories(db: Session) -> List[str]:
    results = db.query(Quote.category).distinct().filter(Quote.category.isnot(None)).all()
    return [r[0] for r in results]

def get_all_tags(db: Session) -> List[str]:
    quotes = db.query(Quote.tags).filter(Quote.tags.isnot(None)).all()
    all_tags = []
    for q in quotes:
        if q[0]:
            all_tags.extend([t.strip() for t in q[0].split(",")])
    return list(set(all_tags))

Writing app/crud.py


In [ ]:
%%writefile app/crawler.py
import httpx
from bs4 import BeautifulSoup
from typing import List, Dict
import time

BASE_URL = "[quotes.toscrape.com](https://quotes.toscrape.com)"

# 카테고리(태그) 목록
CATEGORIES = [
    "love", "inspirational", "life", "humor", "books",
    "reading", "friendship", "success", "motivation", "truth"
]

def crawl_quotes_by_category(category: str, limit: int = 20) -> List[Dict]:
    """특정 카테고리에서 격언 수집"""
    quotes = []
    page = 1

    while len(quotes) < limit:
        url = f"{BASE_URL}/tag/{category}/page/{page}/"

        try:
            response = httpx.get(url, timeout=10.0)
            if response.status_code != 200:
                break

            soup = BeautifulSoup(response.text, "html.parser")
            quote_divs = soup.find_all("div", class_="quote")

            if not quote_divs:
                break

            for div in quote_divs:
                if len(quotes) >= limit:
                    break

                text_elem = div.find("span", class_="text")
                author_elem = div.find("small", class_="author")
                tag_elems = div.find_all("a", class_="tag")

                if text_elem and author_elem:
                    quote_text = text_elem.get_text(strip=True)
                    # 따옴표 제거
                    quote_text = quote_text.strip("\"\").strip("\"\").strip('"')

                    quotes.append({
                        "text": quote_text,
                        "author": author_elem.get_text(strip=True),
                        "tags": ", ".join([t.get_text(strip=True) for t in tag_elems]),
                        "category": category
                    })

            page += 1
            time.sleep(0.5)  # 예의 바른 크롤링

        except Exception as e:
            print(f"Error crawling {category}: {e}")
            break

    return quotes[:limit]

def crawl_all_categories(quotes_per_category: int = 20) -> List[Dict]:
    """모든 카테고리에서 격언 수집"""
    all_quotes = []

    for category in CATEGORIES:
        print(f"Crawling category: {category}")
        quotes = crawl_quotes_by_category(category, quotes_per_category)
        all_quotes.extend(quotes)
        print(f"  - Collected {len(quotes)} quotes")

    return all_quotes

Writing app/crawler.py


In [ ]:
%%writefile app/main.py
from fastapi import FastAPI, Depends, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from sqlalchemy.orm import Session
from typing import Optional, List
from collections import Counter
import re

import gradio as gr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from wordcloud import WordCloud
import pandas as pd
import io
import base64

from app.database import engine, get_db, SessionLocal, Base
from app.models import Quote
from app.schemas import QuoteCreate, QuoteUpdate, QuoteResponse, QuoteList
from app import crud
from app.crawler import crawl_all_categories, CATEGORIES

# 데이터베이스 테이블 생성 (여기서는 주석 처리하고 별도 셀에서 실행)
# Base.metadata.create_all(bind=engine)

# FastAPI 앱 생성
app = FastAPI(
    title="격언 관리 시스템",
    description="격언을 수집, 저장, 분석하는 API",
    version="1.0.0"
)

# CORS 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# =====================
# REST API Endpoints
# =====================

@app.get("/", tags=["Root"])
def read_root():
    return {
        "message": "격언 관리 시스템 API",
        "docs": "/docs",
        "gradio_ui": "/ui"
    }

@app.post("/api/crawl", tags=["Crawler"])
def crawl_and_store(quotes_per_category: int = 20):
    """웹에서 격언을 크롤링하여 DB에 저장"""
    db = SessionLocal()
    try:
        quotes_data = crawl_all_categories(quotes_per_category)

        stored_count = 0
        for q in quotes_data:
            # 중복 체크
            existing = db.query(Quote).filter(
                Quote.text == q["text"],
                Quote.author == q["author"]
            ).first()

            if not existing:
                quote = QuoteCreate(**q)
                crud.create_quote(db, quote)
                stored_count += 1

        return {
            "message": "크롤링 완료",
            "crawled": len(quotes_data),
            "stored": stored_count,
            "categories": CATEGORIES
        }
    finally:
        db.close()

@app.post("/api/quotes", response_model=QuoteResponse, tags=["Quotes"])
def create_quote(quote: QuoteCreate, db: Session = Depends(get_db)):
    """새 격언 생성"""
    return crud.create_quote(db, quote)

@app.get("/api/quotes", response_model=QuoteList, tags=["Quotes"])
def read_quotes(
    skip: int = Query(0, ge=0),
    limit: int = Query(100, ge=1, le=500),
    author: Optional[str] = None,
    category: Optional[str] = None,
    search: Optional[str] = None,
    db: Session = Depends(get_db)
):
    """격언 목록 조회 (필터링, 검색, 페이지네이션 지원)"""
    quotes = crud.get_quotes(db, skip, limit, author, category, search)
    total = crud.get_quotes_count(db, author, category, search)
    return QuoteList(total=total, quotes=quotes)

@app.get("/api/quotes/{quote_id}", response_model=QuoteResponse, tags=["Quotes"])
def read_quote(quote_id: int, db: Session = Depends(get_db)):
    """특정 격언 조회"""
    quote = crud.get_quote(db, quote_id)
    if not quote:
        raise HTTPException(status_code=404, detail="Quote not found")
    return quote

@app.put("/api/quotes/{quote_id}", response_model=QuoteResponse, tags=["Quotes"])
def update_quote(quote_id: int, quote: QuoteUpdate, db: Session = Depends(get_db)):
    """격언 수정"""
    updated = crud.update_quote(db, quote_id, quote)
    if not updated:
        raise HTTPException(status_code=404, detail="Quote not found")
    return updated

@app.delete("/api/quotes/{quote_id}", tags=["Quotes"])
def delete_quote(quote_id: int, db: Session = Depends(get_db)):
    """격언 삭제"""
    if not crud.delete_quote(db, quote_id):
        raise HTTPException(status_code=404, detail="Quote not found")
    return {"message": "Quote deleted successfully"}

@app.get("/api/stats", tags=["Statistics"])
def get_statistics(db: Session = Depends(get_db)):
    """기본 통계 정보"""
    quotes = crud.get_quotes(db, limit=10000)

    # 저자별 카운트
    author_counts = Counter(q.author for q in quotes)

    # 카테고리별 카운트
    category_counts = Counter(q.category for q in quotes if q.category)

    # 태그별 카운트
    all_tags = []
    for q in quotes:
        if q.tags:
            all_tags.extend([t.strip() for t in q.tags.split(",")])
    tag_counts = Counter(all_tags)

    return {
        "total_quotes": len(quotes),
        "unique_authors": len(author_counts),
        "top_authors": author_counts.most_common(10),
        "category_distribution": dict(category_counts),
        "top_tags": tag_counts.most_common(20)
    }

@app.get("/api/authors", tags=["Metadata"])
def get_authors(db: Session = Depends(get_db)):
    """모든 저자 목록"""
    return crud.get_all_authors(db)

@app.get("/api/categories", tags=["Metadata"])
def get_categories(db: Session = Depends(get_db)):
    """모든 카테고리 목록"""
    return crud.get_all_categories(db)

# =====================
# Gradio UI
# =====================

def get_db_session():
    return SessionLocal()

def fetch_quotes_for_display(author_filter="", category_filter="", search=""):
    db = get_db_session()
    try:
        quotes = crud.get_quotes(
            db, limit=500,
            author=author_filter if author_filter else None,
            category=category_filter if category_filter else None,
            search=search if search else None
        )

        data = []
        for q in quotes:
            data.append([
                q.id,
                q.text[:100] + "..." if len(q.text) > 100 else q.text,
                q.author,
                q.category or "",
                q.tags or ""
            ])

        return pd.DataFrame(data, columns=["ID", "Text", "Author", "Category", "Tags"])
    finally:
        db.close()

def run_crawler(quotes_per_cat):
    db = get_db_session()
    try:
        from app.crawler import crawl_all_categories
        quotes_data = crawl_all_categories(int(quotes_per_cat))

        stored = 0
        for q in quotes_data:
            existing = db.query(Quote).filter(
                Quote.text == q["text"],
                Quote.author == q["author"]
            ).first()
            if not existing:
                crud.create_quote(db, QuoteCreate(**q))
                stored += 1

        return f"✅ 크롤링 완료! 수집: {len(quotes_data)}개, 저장: {stored}개"
    finally:
        db.close()

def add_quote(text, author, category, tags):
    if not text or not author:
        return "❌ 텍스트와 저자는 필수입니다."

    db = get_db_session()
    try:
        quote = QuoteCreate(text=text, author=author, category=category, tags=tags)
        crud.create_quote(db, quote)
        return "✅ 격언이 추가되었습니다."
    finally:
        db.close()

def delete_quote_by_id(quote_id):
    if not quote_id:
        return "❌ ID를 입력하세요."

    db = get_db_session()
    try:
        if crud.delete_quote(db, int(quote_id)):
            return f"✅ ID {quote_id} 격언이 삭제되었습니다."
        return "❌ 해당 ID의 격언을 찾을 수 없습니다."
    finally:
        db.close()

def generate_word_frequency():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)

        all_text = " ".join([q.text for q in quotes])
        # 단어 추출 (영문 기준)
        words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())

        # 불용어 제거
        stopwords = {'the', 'and', 'that', 'this', 'with', 'you', 'are', 'for',
                     'not', 'but', 'have', 'was', 'were', 'they', 'from', 'your',
                     'can', 'will', 'all', 'has', 'been', 'would', 'there', 'their'}
        words = [w for w in words if w not in stopwords]

        word_counts = Counter(words)
        top_words = word_counts.most_common(30)

        # 막대 그래프
        fig, ax = plt.subplots(figsize=(12, 6))
        words_list = [w[0] for w in top_words]
        counts_list = [w[1] for w in top_words]

        ax.barh(words_list[::-1], counts_list[::-1], color='steelblue')
        ax.set_xlabel('Frequency')
        ax.set_title('Top 30 Words in Quotes')
        plt.tight_layout()

        return fig
    finally:
        db.close()

def generate_wordcloud():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        all_text = " ".join([q.text for q in quotes])

        if not all_text.strip():
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', fontsize=20)
            ax.axis('off')
            return fig

        wordcloud = WordCloud(
            width=1200, height=600,
            background_color='white',
            colormap='viridis',
            max_words=100
        ).generate(all_text)

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(wordcloud, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Quote Word Cloud')

        return fig
    finally:
        db.close()

def generate_author_chart():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        author_counts = Counter(q.author for q in quotes)
        top_authors = author_counts.most_common(15)

        fig, ax = plt.subplots(figsize=(10, 6))
        authors = [a[0] for a in top_authors]
        counts = [a[1] for a in top_authors]

        ax.barh(authors[::-1], counts[::-1], color='coral')
        ax.set_xlabel('Number of Quotes')
        ax.set_title('Top 15 Authors by Quote Count')
        plt.tight_layout()

        return fig
    finally:
        db.close()

def generate_category_chart():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        category_counts = Counter(q.category for q in quotes if q.category)

        fig, ax = plt.subplots(figsize=(10, 6))
        categories = list(category_counts.keys())
        counts = list(category_counts.values())

        colors = plt.cm.Pastel1(range(len(categories)))
        ax.pie(counts, labels=categories, autopct='%1.1f%%', colors=colors)
        ax.set_title('Quote Distribution by Category')

        return fig
    finally:
        db.close()

def generate_tag_network():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)

        all_tags = []
        for q in quotes:
            if q.tags:
                all_tags.extend([t.strip() for t in q.tags.split(",")])

        tag_counts = Counter(all_tags)
        top_tags = tag_counts.most_common(20)

        fig, ax = plt.subplots(figsize=(10, 6))
        tags = [t[0] for t in top_tags]
        counts = [t[1] for t in top_tags]

        ax.barh(tags[::-1], counts[::-1], color='mediumseagreen')
        ax.set_xlabel('Frequency')
        ax.set_title('Top 20 Tags')
        plt.tight_layout()

        return fig
    finally:
        db.close()

def get_quote_length_analysis():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        lengths = [len(q.text) for q in quotes]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # 히스토그램
        axes[0].hist(lengths, bins=30, color='skyblue', edgecolor='black')
        axes[0].set_xlabel('Quote Length (characters)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Distribution of Quote Lengths')

        # 박스플롯
        axes[1].boxplot(lengths, vert=True)
        axes[1].set_ylabel('Length (characters)')
        axes[1].set_title('Quote Length Statistics')

        plt.tight_layout()
        return fig
    finally:
        db.close()

# Gradio 인터페이스 구성
with gr.Blocks(title="격언 관리 시스템", theme=gr.themes.Soft()) as gradio_app:
    gr.Markdown("# 📚 격언 관리 및 분석 시스템")
    gr.Markdown("FastAPI 백엔드와 연동된 격언 데이터 관리 UI입니다.")

    with gr.Tabs():
        # 탭 1: 데이터 조회
        with gr.TabItem("📋 격언 조회"):
            with gr.Row():
                author_input = gr.Textbox(label="저자 필터", placeholder="저자 이름...")
                category_input = gr.Textbox(label="카테고리 필터", placeholder="카테고리...")
                search_input = gr.Textbox(label="검색", placeholder="키워드 검색...")

            search_btn = gr.Button("🔍 검색", variant="primary")
            quotes_table = gr.Dataframe(
                headers=["ID", "Text", "Author", "Category", "Tags"],
                label="격언 목록"
            )

            search_btn.click(
                fetch_quotes_for_display,
                inputs=[author_input, category_input, search_input],
                outputs=quotes_table
            )

        # 탭 2: 격언 추가
        with gr.TabItem("➕ 격언 추가"):
            new_text = gr.Textbox(label="격언 텍스트", lines=3)
            new_author = gr.Textbox(label="저자")
            new_category = gr.Textbox(label="카테고리")
            new_tags = gr.Textbox(label="태그 (쉼표로 구분)")
            add_btn = gr.Button("추가", variant="primary")
            add_result = gr.Textbox(label="결과")

            add_btn.click(
                add_quote,
                inputs=[new_text, new_author, new_category, new_tags],
                outputs=add_result
            )

        # 탭 3: 격언 삭제
        with gr.TabItem("🗑️ 격언 삭제"):
            delete_id = gr.Number(label="삭제할 격언 ID", precision=0)
            delete_btn = gr.Button("삭제", variant="stop")
            delete_result = gr.Textbox(label="결과")

            delete_btn.click(
                delete_quote_by_id,
                inputs=[delete_id],
                outputs=delete_result
            )

        # 탭 4: 크롤링
        with gr.TabItem("🕷️ 크롤링"):
            gr.Markdown("**quotes.toscrape.com**에서 격언을 수집합니다.")
            crawl_count = gr.Slider(5, 50, value=20, step=5, label="카테고리당 수집 개수")
            crawl_btn = gr.Button("크롤링 시작", variant="primary")
            crawl_result = gr.Textbox(label="결과")

            crawl_btn.click(run_crawler, inputs=[crawl_count], outputs=crawl_result)

        # 탭 5: 분석 - 단어 빈도
        with gr.TabItem("📊 단어 빈도 분석"):
            word_freq_btn = gr.Button("단어 빈도 분석 실행", variant="primary")
            word_freq_plot = gr.Plot(label="단어 빈도 차트")

            word_freq_btn.click(generate_word_frequency, outputs=word_freq_plot)

        # 탭 6: 워드클라우드
        with gr.TabItem("☁️ 워드클라우드"):
            wordcloud_btn = gr.Button("워드클라우드 생성", variant="primary")
            wordcloud_plot = gr.Plot(label="워드클라우드")

            wordcloud_btn.click(generate_wordcloud, outputs=wordcloud_plot)

        # 탭 7: 저자 분석
        with gr.TabItem("👤 저자별 분석"):
            author_chart_btn = gr.Button("저자별 통계 생성", variant="primary")
            author_chart_plot = gr.Plot(label="저자별 격언 수")

            author_chart_btn.click(generate_author_chart, outputs=author_chart_plot)

        # 탭 8: 카테고리 분석
        with gr.TabItem("📁 카테고리 분석"):
            category_chart_btn = gr.Button("카테고리 분포 생성", variant="primary")
            category_chart_plot = gr.Plot(label="카테고리 분포")

            category_chart_btn.click(generate_category_chart, outputs=category_chart_plot)

        # 탭 9: 태그 분석
        with gr.TabItem("🏷️ 태그 분석"):
            tag_btn = gr.Button("태그 분석 실행", variant="primary")
            tag_plot = gr.Plot(label="인기 태그")

            tag_btn.click(generate_tag_network, outputs=tag_plot)

        # 탭 10: 길이 분석
        with gr.TabItem("📏 격언 길이 분석"):
            length_btn = gr.Button("길이 분석 실행", variant="primary")
            length_plot = gr.Plot(label="격언 길이 분포")

            length_btn.click(get_quote_length_analysis, outputs=length_plot)

# Gradio를 FastAPI에 마운트
app = gr.mount_gradio_app(app, gradio_app, path="/ui")

# 서버 실행
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=7860)


Writing app/main.py


After creating the python files in the correct directory structure, we can now import `Base` and `engine` to create the database table.

In [ ]:
from app.database import engine, Base
Base.metadata.create_all(bind=engine)

In [ ]:
from app import crawler
import importlib

# 크롤링할 격언 수 지정 (각 카테고리당)
quotes_to_collect_per_category = 20

print(f"Starting to crawl {quotes_to_collect_per_category} quotes per category...")

# Reload the app.crawler module to apply changes
importlib.reload(crawler)

all_collected_quotes = crawler.crawl_all_categories(quotes_to_collect_per_category)

print(f"Successfully collected a total of {len(all_collected_quotes)} quotes.")

Starting to crawl 20 quotes per category...
Crawling category: love
  - Collected 14 quotes
Crawling category: inspirational
  - Collected 13 quotes
Crawling category: life
  - Collected 13 quotes
Crawling category: humor
  - Collected 12 quotes
Crawling category: books
  - Collected 11 quotes
Crawling category: reading
  - Collected 7 quotes
Crawling category: friendship
  - Collected 5 quotes
Crawling category: success
  - Collected 1 quotes
Crawling category: motivation
  - Collected 0 quotes
Crawling category: truth
  - Collected 4 quotes
Successfully collected a total of 80 quotes.


In [ ]:
from app.database import SessionLocal
from app import crud

db = SessionLocal()
try:
    total_quotes_in_db = crud.get_quotes_count(db)
    print(f"현재 데이터베이스에 저장된 총 격언 수: {total_quotes_in_db}개")
finally:
    db.close()

현재 데이터베이스에 저장된 총 격언 수: 0개


In [ ]:
from pydantic import BaseModel
from datetime import datetime
from typing import Optional, List

class QuoteBase(BaseModel):
    text: str
    author: str
    tags: Optional[str] = None
    category: Optional[str] = None

class QuoteCreate(QuoteBase):
    pass

class QuoteUpdate(BaseModel):
    text: Optional[str] = None
    author: Optional[str] = None
    tags: Optional[str] = None
    category: Optional[str] = None

class QuoteResponse(QuoteBase):
    id: int
    created_at: datetime
    updated_at: Optional[datetime] = None

    class Config:
        from_attributes = True

class QuoteList(BaseModel):
    total: int
    quotes: List[QuoteResponse]


/tmp/ipykernel_1983/3900789802.py:20: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class QuoteResponse(QuoteBase):


In [ ]:
from sqlalchemy.orm import Session
from sqlalchemy import or_
from app.models import Quote
from app.schemas import QuoteCreate, QuoteUpdate
from typing import Optional, List

def create_quote(db: Session, quote: QuoteCreate) -> Quote:
    db_quote = Quote(**quote.model_dump())
    db.add(db_quote)
    db.commit()
    db.refresh(db_quote)
    return db_quote

def get_quote(db: Session, quote_id: int) -> Optional[Quote]:
    return db.query(Quote).filter(Quote.id == quote_id).first()

def get_quotes(
    db: Session,
    skip: int = 0,
    limit: int = 100,
    author: Optional[str] = None,
    category: Optional[str] = None,
    search: Optional[str] = None
) -> List[Quote]:
    query = db.query(Quote)

    if author:
        query = query.filter(Quote.author.ilike(f"%{author}%"))
    if category:
        query = query.filter(Quote.category == category)
    if search:
        query = query.filter(
            or_(
                Quote.text.ilike(f"%{search}%"),
                Quote.author.ilike(f"%{search}%"),
                Quote.tags.ilike(f"%{search}%")
            )
        )

    return query.offset(skip).limit(limit).all()

def get_quotes_count(
    db: Session,
    author: Optional[str] = None,
    category: Optional[str] = None,
    search: Optional[str] = None
) -> int:
    query = db.query(Quote)

    if author:
        query = query.filter(Quote.author.ilike(f"%{author}%"))
    if category:
        query = query.filter(Quote.category == category)
    if search:
        query = query.filter(
            or_(
                Quote.text.ilike(f"%{search}%"),
                Quote.author.ilike(f"%{search}%"),
                Quote.tags.ilike(f"%{search}%")
            )
        )

    return query.count()

def update_quote(db: Session, quote_id: int, quote: QuoteUpdate) -> Optional[Quote]:
    db_quote = db.query(Quote).filter(Quote.id == quote_id).first()
    if db_quote:
        update_data = quote.model_dump(exclude_unset=True)
        for key, value in update_data.items():
            setattr(db_quote, key, value)
        db.commit()
        db.refresh(db_quote)
    return db_quote

def delete_quote(db: Session, quote_id: int) -> bool:
    db_quote = db.query(Quote).filter(Quote.id == quote_id).first()
    if db_quote:
        db.delete(db_quote)
        db.commit()
        return True
    return False

def get_all_authors(db: Session) -> List[str]:
    results = db.query(Quote.author).distinct().all()
    return [r[0] for r in results]

def get_all_categories(db: Session) -> List[str]:
    results = db.query(Quote.category).distinct().filter(Quote.category.isnot(None)).all()
    return [r[0] for r in results]

def get_all_tags(db: Session) -> List[str]:
    quotes = db.query(Quote.tags).filter(Quote.tags.isnot(None)).all()
    all_tags = []
    for q in quotes:
        if q[0]:
            all_tags.extend([t.strip() for t in q[0].split(",")])
    return list(set(all_tags))


In [ ]:
%%writefile app/crawler.py
import httpx
from bs4 import BeautifulSoup
from typing import List, Dict
import time

BASE_URL = "https://quotes.toscrape.com"

# 카테고리(태그) 목록
CATEGORIES = [
    "love", "inspirational", "life", "humor", "books",
    "reading", "friendship", "success", "motivation", "truth"
]

def crawl_quotes_by_category(category: str, limit: int = 20) -> List[Dict]:
    """특정 카테고리에서 격언 수집"""
    quotes = []
    page = 1

    while len(quotes) < limit:
        url = f"{BASE_URL}/tag/{category}/page/{page}/"

        try:
            response = httpx.get(url, timeout=10.0)
            if response.status_code != 200:
                break

            soup = BeautifulSoup(response.text, "html.parser")
            quote_divs = soup.find_all("div", class_="quote")

            if not quote_divs:
                break

            for div in quote_divs:
                if len(quotes) >= limit:
                    break

                text_elem = div.find("span", class_="text")
                author_elem = div.find("small", class_="author")
                tag_elems = div.find_all("a", class_="tag")

                if text_elem and author_elem:
                    quote_text = text_elem.get_text(strip=True)
                    # 따옴표 제거
                    quote_text = quote_text.strip('"\'') # Correctly strip both single and double quotes

                    quotes.append({
                        "text": quote_text,
                        "author": author_elem.get_text(strip=True),
                        "tags": ", ".join([t.get_text(strip=True) for t in tag_elems]),
                        "category": category
                    })

            page += 1
            time.sleep(0.5)  # 예의 바른 크롤링

        except Exception as e:
            print(f"Error crawling {category}: {e}")
            break

    return quotes[:limit]

def crawl_all_categories(quotes_per_category: int = 20) -> List[Dict]:
    """모든 카테고리에서 격언 수집"""
    all_quotes = []

    for category in CATEGORIES:
        print(f"Crawling category: {category}")
        quotes = crawl_quotes_by_category(category, quotes_per_category)
        all_quotes.extend(quotes)
        print(f"  - Collected {len(quotes)} quotes")

    return all_quotes

Overwriting app/crawler.py


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from sqlalchemy.orm import Session
from typing import Optional, List
from collections import Counter
import re

import gradio as gr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from wordcloud import WordCloud
import pandas as pd
import io
import base64

from app.database import engine, get_db, SessionLocal, Base # Removed Base from here, it's defined in app.database
from app.models import Quote
from app.schemas import QuoteCreate, QuoteUpdate, QuoteResponse, QuoteList
from app import crud
from app.crawler import crawl_all_categories, CATEGORIES

# 데이터베이스 테이블 생성
Base.metadata.create_all(bind=engine)

# FastAPI 앱 생성
app = FastAPI(
    title="격언 관리 시스템",
    description="격언을 수집, 저장, 분석하는 API",
    version="1.0.0"
)

# CORS 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# =====================
# REST API Endpoints
# =====================

@app.get("/", tags=["Root"])
def read_root():
    return {
        "message": "격언 관리 시스템 API",
        "docs": "/docs",
        "gradio_ui": "/ui"
    }

@app.post("/api/crawl", tags=["Crawler"])
def crawl_and_store(quotes_per_category: int = 20):
    """웹에서 격언을 크롤링하여 DB에 저장"""
    db = SessionLocal()
    try:
        quotes_data = crawl_all_categories(quotes_per_category)

        stored_count = 0
        for q in quotes_data:
            # 중복 체크
            existing = db.query(Quote).filter(
                Quote.text == q["text"],
                Quote.author == q["author"]
            ).first()

            if not existing:
                quote = QuoteCreate(**q)
                crud.create_quote(db, quote)
                stored_count += 1

        return {
            "message": "크롤링 완료",
            "crawled": len(quotes_data),
            "stored": stored_count,
            "categories": CATEGORIES
        }
    finally:
        db.close()

@app.post("/api/quotes", response_model=QuoteResponse, tags=["Quotes"])
def create_quote(quote: QuoteCreate, db: Session = Depends(get_db)):
    """새 격언 생성"""
    return crud.create_quote(db, quote)

@app.get("/api/quotes", response_model=QuoteList, tags=["Quotes"])
def read_quotes(
    skip: int = Query(0, ge=0),
    limit: int = Query(100, ge=1, le=500),
    author: Optional[str] = None,
    category: Optional[str] = None,
    search: Optional[str] = None,
    db: Session = Depends(get_db)
):
    """격언 목록 조회 (필터링, 검색, 페이지네이션 지원)"""
    quotes = crud.get_quotes(db, skip, limit, author, category, search)
    total = crud.get_quotes_count(db, author, category, search)
    return QuoteList(total=total, quotes=quotes)

@app.get("/api/quotes/{quote_id}", response_model=QuoteResponse, tags=["Quotes"])
def read_quote(quote_id: int, db: Session = Depends(get_db)):
    """특정 격언 조회"""
    quote = crud.get_quote(db, quote_id)
    if not quote:
        raise HTTPException(status_code=404, detail="Quote not found")
    return quote

@app.put("/api/quotes/{quote_id}", response_model=QuoteResponse, tags=["Quotes"])
def update_quote(quote_id: int, quote: QuoteUpdate, db: Session = Depends(get_db)):
    """격언 수정"""
    updated = crud.update_quote(db, quote_id, quote)
    if not updated:
        raise HTTPException(status_code=404, detail="Quote not found")
    return updated

@app.delete("/api/quotes/{quote_id}", tags=["Quotes"])
def delete_quote(quote_id: int, db: Session = Depends(get_db)):
    """격언 삭제"""
    if not crud.delete_quote(db, quote_id):
        raise HTTPException(status_code=404, detail="Quote not found")
    return {"message": "Quote deleted successfully"}

@app.get("/api/stats", tags=["Statistics"])
def get_statistics(db: Session = Depends(get_db)):
    """기본 통계 정보"""
    quotes = crud.get_quotes(db, limit=10000)

    # 저자별 카운트
    author_counts = Counter(q.author for q in quotes)

    # 카테고리별 카운트
    category_counts = Counter(q.category for q in quotes if q.category)

    # 태그별 카운트
    all_tags = []
    for q in quotes:
        if q.tags:
            all_tags.extend([t.strip() for t in q.tags.split(",")])
    tag_counts = Counter(all_tags)

    return {
        "total_quotes": len(quotes),
        "unique_authors": len(author_counts),
        "top_authors": author_counts.most_common(10),
        "category_distribution": dict(category_counts),
        "top_tags": tag_counts.most_common(20)
    }

@app.get("/api/authors", tags=["Metadata"])
def get_authors(db: Session = Depends(get_db)):
    """모든 저자 목록"""
    return crud.get_all_authors(db)

@app.get("/api/categories", tags=["Metadata"])
def get_categories(db: Session = Depends(get_db)):
    """모든 카테고리 목록"""
    return crud.get_all_categories(db)

# =====================
# Gradio UI
# =====================

def get_db_session():
    return SessionLocal()

def fetch_quotes_for_display(author_filter="", category_filter="", search=""):
    db = get_db_session()
    try:
        quotes = crud.get_quotes(
            db, limit=500,
            author=author_filter if author_filter else None,
            category=category_filter if category_filter else None,
            search=search if search else None
        )

        data = []
        for q in quotes:
            data.append([
                q.id,
                q.text[:100] + "..." if len(q.text) > 100 else q.text,
                q.author,
                q.category or "",
                q.tags or ""
            ])

        return pd.DataFrame(data, columns=["ID", "Text", "Author", "Category", "Tags"])
    finally:
        db.close()

def run_crawler(quotes_per_cat):
    db = get_db_session()
    try:
        from app.crawler import crawl_all_categories
        quotes_data = crawl_all_categories(int(quotes_per_cat))

        stored = 0
        for q in quotes_data:
            existing = db.query(Quote).filter(
                Quote.text == q["text"],
                Quote.author == q["author"]
            ).first()
            if not existing:
                crud.create_quote(db, QuoteCreate(**q))
                stored += 1

        return f"✅ 크롤링 완료! 수집: {len(quotes_data)}개, 저장: {stored}개"
    finally:
        db.close()

def add_quote(text, author, category, tags):
    if not text or not author:
        return "❌ 텍스트와 저자는 필수입니다."

    db = get_db_session()
    try:
        quote = QuoteCreate(text=text, author=author, category=category, tags=tags)
        crud.create_quote(db, quote)
        return "✅ 격언이 추가되었습니다."
    finally:
        db.close()

def delete_quote_by_id(quote_id):
    if not quote_id:
        return "❌ ID를 입력하세요."

    db = get_db_session()
    try:
        if crud.delete_quote(db, int(quote_id)):
            return f"✅ ID {quote_id} 격언이 삭제되었습니다."
        return "❌ 해당 ID의 격언을 찾을 수 없습니다."
    finally:
        db.close()

def generate_word_frequency():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)

        all_text = " ".join([q.text for q in quotes])
        # 단어 추출 (영문 기준)
        words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())

        # 불용어 제거
        stopwords = {'the', 'and', 'that', 'this', 'with', 'you', 'are', 'for',
                     'not', 'but', 'have', 'was', 'were', 'they', 'from', 'your',
                     'can', 'will', 'all', 'has', 'been', 'would', 'there', 'their'}
        words = [w for w in words if w not in stopwords]

        word_counts = Counter(words)
        top_words = word_counts.most_common(30)

        # 막대 그래프
        fig, ax = plt.subplots(figsize=(12, 6))
        words_list = [w[0] for w in top_words]
        counts_list = [w[1] for w in top_words]

        ax.barh(words_list[::-1], counts_list[::-1], color='steelblue')
        ax.set_xlabel('Frequency')
        ax.set_title('Top 30 Words in Quotes')
        plt.tight_layout()

        return fig
    finally:
        db.close()

def generate_wordcloud():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        all_text = " ".join([q.text for q in quotes])

        if not all_text.strip():
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', fontsize=20)
            ax.axis('off')
            return fig

        wordcloud = WordCloud(
            width=1200, height=600,
            background_color='white',
            colormap='viridis',
            max_words=100
        ).generate(all_text)

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(wordcloud, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Quote Word Cloud')

        return fig
    finally:
        db.close()

def generate_author_chart():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        author_counts = Counter(q.author for q in quotes)
        top_authors = author_counts.most_common(15)

        fig, ax = plt.subplots(figsize=(10, 6))
        authors = [a[0] for a in top_authors]
        counts = [a[1] for a in top_authors]

        ax.barh(authors[::-1], counts[::-1], color='coral')
        ax.set_xlabel('Number of Quotes')
        ax.set_title('Top 15 Authors by Quote Count')
        plt.tight_layout()

        return fig
    finally:
        db.close()

def generate_category_chart():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        category_counts = Counter(q.category for q in quotes if q.category)

        fig, ax = plt.subplots(figsize=(10, 6))
        categories = list(category_counts.keys())
        counts = list(category_counts.values())

        colors = plt.cm.Pastel1(range(len(categories)))
        ax.pie(counts, labels=categories, autopct='%1.1f%%', colors=colors)
        ax.set_title('Quote Distribution by Category')

        return fig
    finally:
        db.close()

def generate_tag_network():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)

        all_tags = []
        for q in quotes:
            if q.tags:
                all_tags.extend([t.strip() for t in q.tags.split(",")])

        tag_counts = Counter(all_tags)
        top_tags = tag_counts.most_common(20)

        fig, ax = plt.subplots(figsize=(10, 6))
        tags = [t[0] for t in top_tags]
        counts = [t[1] for t in top_tags]

        ax.barh(tags[::-1], counts[::-1], color='mediumseagreen')
        ax.set_xlabel('Frequency')
        ax.set_title('Top 20 Tags')
        plt.tight_layout()

        return fig
    finally:
        db.close()

def get_quote_length_analysis():
    db = get_db_session()
    try:
        quotes = crud.get_quotes(db, limit=10000)
        lengths = [len(q.text) for q in quotes]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # 히스토그램
        axes[0].hist(lengths, bins=30, color='skyblue', edgecolor='black')
        axes[0].set_xlabel('Quote Length (characters)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Distribution of Quote Lengths')

        # 박스플롯
        axes[1].boxplot(lengths, vert=True)
        axes[1].set_ylabel('Length (characters)')
        axes[1].set_title('Quote Length Statistics')

        plt.tight_layout()
        return fig
    finally:
        db.close()

# Gradio 인터페이스 구성
with gr.Blocks(title="격언 관리 시스템", theme=gr.themes.Soft()) as gradio_app:
    gr.Markdown("# 📚 격언 관리 및 분석 시스템")
    gr.Markdown("FastAPI 백엔드와 연동된 격언 데이터 관리 UI입니다.")

    with gr.Tabs():
        # 탭 1: 데이터 조회
        with gr.TabItem("📋 격언 조회"):
            with gr.Row():
                author_input = gr.Textbox(label="저자 필터", placeholder="저자 이름...")
                category_input = gr.Textbox(label="카테고리 필터", placeholder="카테고리...")
                search_input = gr.Textbox(label="검색", placeholder="키워드 검색...")

            search_btn = gr.Button("🔍 검색", variant="primary")
            quotes_table = gr.Dataframe(
                headers=["ID", "Text", "Author", "Category", "Tags"],
                label="격언 목록"
            )

            search_btn.click(
                fetch_quotes_for_display,
                inputs=[author_input, category_input, search_input],
                outputs=quotes_table
            )

        # 탭 2: 격언 추가
        with gr.TabItem("➕ 격언 추가"):
            new_text = gr.Textbox(label="격언 텍스트", lines=3)
            new_author = gr.Textbox(label="저자")
            new_category = gr.Textbox(label="카테고리")
            new_tags = gr.Textbox(label="태그 (쉼표로 구분)")
            add_btn = gr.Button("추가", variant="primary")
            add_result = gr.Textbox(label="결과")

            add_btn.click(
                add_quote,
                inputs=[new_text, new_author, new_category, new_tags],
                outputs=add_result
            )

        # 탭 3: 격언 삭제
        with gr.TabItem("🗑️ 격언 삭제"):
            delete_id = gr.Number(label="삭제할 격언 ID", precision=0)
            delete_btn = gr.Button("삭제", variant="stop")
            delete_result = gr.Textbox(label="결과")

            delete_btn.click(
                delete_quote_by_id,
                inputs=[delete_id],
                outputs=delete_result
            )

        # 탭 4: 크롤링
        with gr.TabItem("🕷️ 크롤링"):
            gr.Markdown("**quotes.toscrape.com**에서 격언을 수집합니다.")
            crawl_count = gr.Slider(5, 50, value=20, step=5, label="카테고리당 수집 개수")
            crawl_btn = gr.Button("크롤링 시작", variant="primary")
            crawl_result = gr.Textbox(label="결과")

            crawl_btn.click(run_crawler, inputs=[crawl_count], outputs=crawl_result)

        # 탭 5: 분석 - 단어 빈도
        with gr.TabItem("📊 단어 빈도 분석"):
            word_freq_btn = gr.Button("단어 빈도 분석 실행", variant="primary")
            word_freq_plot = gr.Plot(label="단어 빈도 차트")

            word_freq_btn.click(generate_word_frequency, outputs=word_freq_plot)

        # 탭 6: 워드클라우드
        with gr.TabItem("☁️ 워드클라우드"):
            wordcloud_btn = gr.Button("워드클라우드 생성", variant="primary")
            wordcloud_plot = gr.Plot(label="워드클라우드")

            wordcloud_btn.click(generate_wordcloud, outputs=wordcloud_plot)

        # 탭 7: 저자 분석
        with gr.TabItem("👤 저자별 분석"):
            author_chart_btn = gr.Button("저자별 통계 생성", variant="primary")
            author_chart_plot = gr.Plot(label="저자별 격언 수")

            author_chart_btn.click(generate_author_chart, outputs=author_chart_plot)

        # 탭 8: 카테고리 분석
        with gr.TabItem("📁 카테고리 분석"):
            category_chart_btn = gr.Button("카테고리 분포 생성", variant="primary")
            category_chart_plot = gr.Plot(label="카테고리 분포")

            category_chart_btn.click(generate_category_chart, outputs=category_chart_plot)

        # 탭 9: 태그 분석
        with gr.TabItem("🏷️ 태그 분석"):
            tag_btn = gr.Button("태그 분석 실행", variant="primary")
            tag_plot = gr.Plot(label="인기 태그")

            tag_btn.click(generate_tag_network, outputs=tag_plot)

        # 탭 10: 길이 분석
        with gr.TabItem("📏 격언 길이 분석"):
            length_btn = gr.Button("길이 분석 실행", variant="primary")
            length_plot = gr.Plot(label="격언 길이 분포")

            length_btn.click(get_quote_length_analysis, outputs=length_plot)

# Gradio를 FastAPI에 마운트
app = gr.mount_gradio_app(app, gradio_app, path="/ui")

# 서버 실행
# 이 블록은 Colab 환경에서 Gradio가 이미 서버를 실행하고 있으므로 주석 처리합니다.
# if __name__ == "__main__":
#     import uvicorn
#     uvicorn.run(app, host="0.0.0.0", port=7860)

/tmp/ipykernel_1983/864407771.py:385: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="격언 관리 시스템", theme=gr.themes.Soft()) as gradio_app:


new /ui


In [ ]:
# app.py (Hugging Face Spaces 진입점)
import sys
sys.path.insert(0, '.')

from app.main import app, gradio_app
from app.database import engine
from app.models import Base

# 테이블 생성 (초기 한 번만 실행되도록, 여기서는 주석 처리합니다.)
# Base.metadata.create_all(bind=engine)

# Gradio 앱을 직접 실행 (Spaces에서)
if __name__ == "__main__":
    gradio_app.launch(server_name="0.0.0.0", server_port=7860)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d85c544d7095072be0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from app.database import SessionLocal
from app import crud
from app.models import Quote

db = SessionLocal()
try:
    # 현재 데이터베이스에 있는 모든 격언을 ID 역순으로 가져옵니다.
    # 가장 높은 ID (가장 최근 추가된)부터 삭제하기 위함입니다.
    all_quotes = db.query(Quote).order_by(Quote.id.desc()).all()
    current_count = len(all_quotes)
    print(f"현재 데이터베이스에 저장된 총 격언 수: {current_count}개")

    target_count = 20

    if current_count > target_count:
        quotes_to_delete_count = current_count - target_count

        # 삭제할 격언들의 ID를 가져옵니다 (가장 높은 ID부터).
        ids_to_delete = [quote.id for quote in all_quotes[:quotes_to_delete_count]]

        print(f"가장 최근 추가된 {quotes_to_delete_count}개의 격언을 삭제합니다. (ID: {ids_to_delete})")

        for quote_id in ids_to_delete:
            crud.delete_quote(db, quote_id)

        # 삭제 후 남은 격언 수를 확인합니다.
        remaining_quotes_count = crud.get_quotes_count(db)
        print(f"삭제 후 데이터베이스에 남은 격언 수: {remaining_quotes_count}개")
    else:
        print(f"현재 격언 수가 {current_count}개로, {target_count}개보다 적거나 같습니다. 삭제할 격언이 없습니다.")
finally:
    db.close()

현재 데이터베이스에 저장된 총 격언 수: 61개
가장 최근 추가된 41개의 격언을 삭제합니다. (ID: [61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21])
삭제 후 데이터베이스에 남은 격언 수: 20개


In [ ]:
from app.database import SessionLocal
from app import crud

db = SessionLocal()
try:
    total_quotes_in_db = crud.get_quotes_count(db)
    print(f"현재 데이터베이스에 저장된 총 격언 수: {total_quotes_in_db}개")
finally:
    db.close()

현재 데이터베이스에 저장된 총 격언 수: 20개


In [ ]:
from app.main import run_crawler
from app.database import SessionLocal
from app import crud

# 크롤링할 격언 수 (각 카테고리당)를 20으로 설정
quotes_to_collect_per_category = 20

print(f"Calling run_crawler to store more quotes (max {quotes_to_collect_per_category} per category)...")
result_message = run_crawler(quotes_to_collect_per_category)
print(result_message)

db = SessionLocal()
try:
    total_quotes_in_db = crud.get_quotes_count(db)
    print(f"현재 데이터베이스에 저장된 총 격언 수: {total_quotes_in_db}개")
finally:
    db.close()

Calling run_crawler to store more quotes (max 20 per category)...
Crawling category: love
  - Collected 14 quotes
Crawling category: inspirational
  - Collected 13 quotes
Crawling category: life
  - Collected 13 quotes
Crawling category: humor
  - Collected 12 quotes
Crawling category: books
  - Collected 11 quotes
Crawling category: reading
  - Collected 7 quotes
Crawling category: friendship
  - Collected 5 quotes
Crawling category: success
  - Collected 1 quotes
Crawling category: motivation
  - Collected 0 quotes
Crawling category: truth
  - Collected 4 quotes
✅ 크롤링 완료! 수집: 80개, 저장: 41개
현재 데이터베이스에 저장된 총 격언 수: 61개


In [ ]:
# 크롤러를 다시 실행하여 격언을 추가합니다.
from app.main import run_crawler
from app.database import SessionLocal
from app import crud

quotes_to_collect_per_category = 20 # 각 카테고리당 최대 20개

print(f"크롤러를 다시 실행하여 격언을 수집하고 저장합니다 (각 카테고리당 최대 {quotes_to_collect_per_category}개)...")
result_message = run_crawler(quotes_to_collect_per_category)
print(result_message)

db = SessionLocal()
try:
    total_quotes_after_crawl = crud.get_quotes_count(db)
    print(f"크롤링 후 데이터베이스에 저장된 총 격언 수: {total_quotes_after_crawl}개")

    # 만약 여전히 20개 미만이면, 수동으로 1개를 추가하여 총 20개를 만듭니다.
    if total_quotes_after_crawl < 20:
        print(f"여전히 {total_quotes_after_crawl}개이므로, 20개를 채우기 위해 격언을 추가합니다.")
        from app.main import add_quote
        add_quote(
            text="The only true wisdom is in knowing you know nothing.",
            author="Socrates",
            category="wisdom",
            tags="wisdom, knowledge, philosophy"
        )
        final_total_quotes = crud.get_quotes_count(db)
        print(f"격언 추가 후 데이터베이스에 저장된 총 격언 수: {final_total_quotes}개")
    else:
        final_total_quotes = total_quotes_after_crawl
        print(f"총 격언 수가 {final_total_quotes}개이므로 추가적인 조치가 필요하지 않습니다.")

finally:
    db.close()

크롤러를 다시 실행하여 격언을 수집하고 저장합니다 (각 카테고리당 최대 20개)...
Crawling category: love
  - Collected 14 quotes
Crawling category: inspirational
  - Collected 13 quotes
Crawling category: life
  - Collected 13 quotes
Crawling category: humor
  - Collected 12 quotes
Crawling category: books
  - Collected 11 quotes
Crawling category: reading
  - Collected 7 quotes
Crawling category: friendship
  - Collected 5 quotes
Crawling category: success
  - Collected 1 quotes
Crawling category: motivation
  - Collected 0 quotes
Crawling category: truth
  - Collected 4 quotes
✅ 크롤링 완료! 수집: 80개, 저장: 0개
크롤링 후 데이터베이스에 저장된 총 격언 수: 61개
총 격언 수가 61개이므로 추가적인 조치가 필요하지 않습니다.


In [ ]:
from app.database import SessionLocal
from app import crud

db = SessionLocal()
try:
    total_quotes_in_db = crud.get_quotes_count(db)
    print(f"현재 데이터베이스에 저장된 총 격언 수: {total_quotes_in_db}개")
finally:
    db.close()

현재 데이터베이스에 저장된 총 격언 수: 61개


In [ ]:
from app.main import add_quote
from app.database import SessionLocal

def add_new_quote_programmatically(text: str, author: str, category: str = None, tags: str = None):
    # This function simulates the add_quote logic but can be called directly
    # It retrieves a db session inside, similar to how the Gradio function does.
    db = SessionLocal()
    try:
        from app.schemas import QuoteCreate
        from app import crud

        quote = QuoteCreate(text=text, author=author, category=category, tags=tags)
        crud.create_quote(db, quote)
        return f"✅ 격언이 추가되었습니다: {text}"
    except Exception as e:
        return f"❌ 격언 추가 중 오류 발생: {e}"
    finally:
        db.close()

# 예시: 'motivation' 카테고리에 격언 추가
print(add_new_quote_programmatically(
    text="The only way to do great work is to love what you do.",
    author="Steve Jobs",
    category="motivation",
    tags="motivation, work, passion"
))

print(add_new_quote_programmatically(
    text="Your time is limited, don't waste it living someone else's life.",
    author="Steve Jobs",
    category="motivation",
    tags="motivation, life, time"
))

# 예시: 'success' 카테고리에 격언 추가
print(add_new_quote_programmatically(
    text="Success is not final, failure is not fatal: it is the courage to continue that counts.",
    author="Winston Churchill",
    category="success",
    tags="success, failure, courage"
))

✅ 격언이 추가되었습니다: The only way to do great work is to love what you do.
✅ 격언이 추가되었습니다: Your time is limited, don't waste it living someone else's life.
✅ 격언이 추가되었습니다: Success is not final, failure is not fatal: it is the courage to continue that counts.


In [ ]:
from collections import Counter

# Assuming CATEGORIES and quotes_to_collect_per_category are already defined
# and all_collected_quotes is available from previous execution

category_counts = Counter(q['category'] for q in all_collected_quotes)

print(f"Target quotes per category: {quotes_to_collect_per_category}\n")
print("Categories with less than 20 quotes collected:")
for category in CATEGORIES:
    count = category_counts.get(category, 0)
    if count < quotes_to_collect_per_category:
        print(f"- {category}: {count} quotes")

Target quotes per category: 20

Categories with less than 20 quotes collected:
- love: 14 quotes
- inspirational: 13 quotes
- life: 13 quotes
- humor: 12 quotes
- books: 11 quotes
- reading: 7 quotes
- friendship: 5 quotes
- success: 1 quotes
- motivation: 0 quotes
- truth: 4 quotes
